#### Question:1 Compute column sums of a matrix using CUDA.

In [1]:
import numpy as np
from numba import cuda

In [11]:
A = np.random.randint(
    low=0,
    high=100,
    size=(1024, 1024),
    dtype=np.int32
)

print(A.shape)

(1024, 1024)


In [3]:
@cuda.jit
def column_sum_kernel(matrix, result):

    col = cuda.grid(1)

    if col < matrix.shape[1]:

        total = 0

        for row in range(matrix.shape[0]):
            total += matrix[row, col]

        result[col] = total

In [12]:
rows, cols = A.shape

result = np.zeros(cols, dtype=np.int32)

print("Initial Result Array:")
print(result)

Initial Result Array:
[0 0 0 ... 0 0 0]


In [13]:
d_A = cuda.to_device(A)
d_result = cuda.to_device(result)

In [14]:
threads_per_block = 256

blocks_per_grid = (cols + threads_per_block - 1) // threads_per_block

print("Blocks per Grid:", blocks_per_grid)
print("Threads per Block:", threads_per_block)

Blocks per Grid: 4
Threads per Block: 256


In [15]:
column_sum_kernel[blocks_per_grid, threads_per_block](d_A, d_result)

d:\Python\lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [16]:
result = d_result.copy_to_host()

print("Column Sums:")
print(result)

Column Sums:
[52667 50723 50062 ... 50508 48919 50619]


In [17]:
cpu_result = np.sum(A, axis=0)

print("CPU Result :", cpu_result)
print("GPU Result :", result)

CPU Result : [52667 50723 50062 ... 50508 48919 50619]
GPU Result : [52667 50723 50062 ... 50508 48919 50619]
